# B2-019-attention-transformers — Practice p13 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** scaled-dot-product-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F5-probability`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

Let \(S=q\cdot k=\sum_{r=1}^{d_k}q_rk_r\). For each coordinate, independence of \(q_r\) and \(k_r\) gives \(E[q_rk_r]=E[q_r]E[k_r]=0\), and \(E[q_r^2k_r^2]=E[q_r^2]E[k_r^2]=1\). Thus each product has variance one. Independence across coordinate pairs makes all cross-covariances zero, so \(\operatorname{Var}(S)=\sum_r1=d_k\). Therefore \(\operatorname{Var}(S/\sqrt{d_k})=1\). If coordinates or query/key pairs are correlated, the product means, variances, or cross-covariances need not have these values, so the result can fail.

A concrete correlated counterexample makes the failure visible. Let every query coordinate equal one shared Rademacher variable `Z` and every key coordinate equal an independent shared Rademacher variable `W`. Each coordinate still has mean zero and variance one, and each `q_r` is independent of its paired `k_r`, but the coordinate products are identical: `q_r*k_r=ZW`. Thus `q dot k=d_k*ZW` and its variance is `d_k**2`, not `d_k`.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 0.08
RTOL = 0.08
rng = np.random.default_rng(SEED)
d_k = 64
q = rng.standard_normal((100_000, d_k))
k = rng.standard_normal((100_000, d_k))
raw_scores = np.sum(q * k, axis=1)
scaled_scores = raw_scores / np.sqrt(d_k)
raw_variance = np.var(raw_scores)
scaled_variance = np.var(scaled_scores)
# Correlated-coordinate counterexample: all products share the same Z*W.
z = np.tile(np.array([-1.0, -1.0, 1.0, 1.0]), 25_000)
w = np.tile(np.array([-1.0, 1.0, -1.0, 1.0]), 25_000)
correlated_scores = d_k * z * w
correlated_variance = np.var(correlated_scores)


### Answer check

In [ ]:
assert np.isclose(raw_variance, d_k, atol=ATOL * d_k, rtol=RTOL)
assert np.isclose(scaled_variance, 1.0, atol=ATOL, rtol=RTOL)
assert np.isclose(raw_variance / d_k, scaled_variance, atol=1e-12, rtol=1e-12)
assert np.isclose(correlated_variance, d_k**2, atol=1e-12, rtol=1e-12)